<a href="https://colab.research.google.com/github/BedonViteri/pe-u4-spark-BCEL/blob/Juliana-Emanuel/notebooks/PE_U4_pipeline_spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PE-U4 — Comprobación experimental de la Ley de Amdahl con Apache Spark
## Dominio de referencia: **BCEL — SGA Escuela Provincias Unidas** (Sistema de Gestión Académica)

**Asignatura:** Aplicaciones Distribuidas (ISR-701) — Unidad 4
**Código de actividad:** GA-SUM-05 / PE-U4
**Equipo:** _(completar nombres, CI y rol de cada integrante)_
**PFC de referencia:** BCEL — SGA Escuela Provincias Unidas

### Justificación técnica del dominio elegido (mínimo 100 palabras)

Dentro del sistema distribuido BCEL — SGA Escuela Provincias Unidas, esta práctica toma como referencia específica el **microservicio Docente** (Django REST + gRPC, esquema `sga_docente` en PostgreSQL sobre AWS EC2), responsable de la gestión de asistencias y evaluaciones/calificaciones. A diferencia de los módulos de Secretaría o Soporte —que generan eventos esporádicos como trámites o tickets— el microservicio Docente produce un registro por cada evento de asistencia o evaluación, por cada estudiante y cada curso, en cada sesión de clase. Este patrón de generación de datos es estructuralmente idéntico al de la tabla `studentVle` del dataset OULAD utilizado en esta práctica: un registro por interacción estudiante-curso, con claves hacia el estudiante y el curso correspondiente. La decisión de negocio que se apoyaría en el pipeline implementado es la **detección temprana de estudiantes en riesgo académico** (baja asistencia, bajo rendimiento recurrente, sobrecarga de créditos matriculados), análisis que hoy el microservicio Docente no ejecuta en modo batch a gran escala, limitándose a consultas transaccionales síncronas sobre `sga_docente`. A medida que BCEL acumule varios períodos académicos con miles de estudiantes activos, el volumen de registros de asistencia y calificaciones superará la capacidad de procesamiento eficiente de consultas SQL síncronas o de pandas en memoria, justificando así un pipeline distribuido con Apache Spark para el análisis histórico y agregado de esta información, ejecutado de forma separada de la base transaccional operativa del microservicio.

### Dataset elegido: OULAD (Open University Learning Analytics Dataset)

- **Fuente:** The Open University (Reino Unido) — Knowledge Media Institute
- **URL de descarga:** http://schools.stem.open.ac.uk/cdn/files/anonymisedData.zip
- **Página oficial:** https://research.stem.open.ac.uk/ouanalyse/dataset/
- **Licencia:** CC-BY 4.0
- **Cita académica:** Kuzilek, J., Hlosta, M., Zdrahal, Z. "Open University Learning Analytics dataset." *Nature Scientific Data* 4, 170171 (2017). doi: 10.1038/sdata.2017.171
- **Registros:** 10,655,280 en `studentVle.csv` (piso institucional de 500,000 superado ampliamente)
- **Tablas usadas:** `studentVle`, `studentInfo`, `vle`


In [8]:
# Instalación (ejecutar solo en Colab/Databricks si no está preinstalado)
!pip install pyspark==3.5.0 -q


In [9]:
import time
import statistics
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import os

os.makedirs("data/pandas", exist_ok=True)
os.makedirs("data/spark", exist_ok=True)
os.makedirs("resultados/figuras", exist_ok=True)
os.makedirs("evidencia", exist_ok=True)


## Paso 1 — Preparación del entorno y del conjunto de datos (peso 7 %)

Descarga del dataset real OULAD y verificación de número de registros, esquema y tamaño.


In [10]:
# Descarga y descompresión del dataset OULAD (ejecutar una sola vez)
!wget -q http://schools.stem.open.ac.uk/cdn/files/anonymisedData.zip -O oulad.zip
!mkdir -p oulad_raw
!unzip -o -q oulad.zip -d oulad_raw
!ls -la oulad_raw


total 453512
drwxr-xr-x 2 root root      4096 Aug  9 00:58 .
drwxr-xr-x 1 root root      4096 Aug  9 00:46 ..
-rw-r--r-- 1 root root      8200 Sep 25  2015 assessments.csv
-rw-r--r-- 1 root root       526 Sep 25  2015 courses.csv
-rw-r--r-- 1 root root   5690310 Sep 25  2015 studentAssessment.csv
-rw-r--r-- 1 root root   3461652 Sep 25  2015 studentInfo.csv
-rw-r--r-- 1 root root   1109984 Sep 25  2015 studentRegistration.csv
-rw-r--r-- 1 root root 453836331 Sep 25  2015 studentVle.csv
-rw-r--r-- 1 root root    260126 Sep 25  2015 vle.csv


In [11]:
# Carga en pandas y verificación de esquema/tamaño
studentVle_pd = pd.read_csv("oulad_raw/studentVle.csv")
studentInfo_pd = pd.read_csv("oulad_raw/studentInfo.csv")
vle_pd = pd.read_csv("oulad_raw/vle.csv")

print("studentVle:", studentVle_pd.shape)
print("studentInfo:", studentInfo_pd.shape)
print("vle:", vle_pd.shape)
print()
print(studentVle_pd.dtypes)


studentVle: (10655280, 6)
studentInfo: (32593, 12)
vle: (6364, 6)

code_module          object
code_presentation    object
id_student            int64
id_site               int64
date                  int64
sum_click             int64
dtype: object


### Tabla de documentación del dataset (para incluir en el informe LaTeX con `booktabs`)

| Fuente | URL | Licencia | Fecha de descarga | N.° registros (studentVle) | N.° columnas | Tamaño en disco |
|---|---|---|---|---|---|---|
| The Open University (OULAD) | http://schools.stem.open.ac.uk/cdn/files/anonymisedData.zip | CC-BY 4.0 | _(completar con fecha real)_ | 10,655,280 | 6 | ~300 MB (zip completo) |


In [12]:
# Cargar también en PySpark y verificar equivalencia de esquema/conteo
spark = (
    SparkSession.builder
    .appName("PE-U4-BCEL-SGA")
    .config("spark.executor.instances", "4")
    .getOrCreate()
)

studentVle_sp = spark.read.csv("oulad_raw/studentVle.csv", header=True, inferSchema=True)
studentInfo_sp = spark.read.csv("oulad_raw/studentInfo.csv", header=True, inferSchema=True)
vle_sp = spark.read.csv("oulad_raw/vle.csv", header=True, inferSchema=True)

print("studentVle (Spark):", studentVle_sp.count())
studentVle_sp.printSchema()

print("Configuración efectiva de la sesión Spark:")
for k, v in spark.sparkContext.getConf().getAll():
    print(f"  {k} = {v}")


studentVle (Spark): 10655280
root
 |-- code_module: string (nullable = true)
 |-- code_presentation: string (nullable = true)
 |-- id_student: integer (nullable = true)
 |-- id_site: integer (nullable = true)
 |-- date: integer (nullable = true)
 |-- sum_click: integer (nullable = true)

Configuración efectiva de la sesión Spark:
  spark.executor.instances = 4
  spark.master = local[4]
  spark.driver.memory = 4g
  spark.app.name = PE-U4-BCEL-SGA
  spark.executor.id = driver
  spark.sql.warehouse.dir = file:/content/spark-warehouse
  spark.app.startTime = 1786236870980
  spark.driver.extraJavaOptions = -Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --

## Paso 2 — Implementación secuencial con pandas (peso 6 %)

Cinco transformaciones T1–T5 (ver `src/transformaciones_pandas.py`), medidas con
`time.perf_counter()`: 5 repeticiones + 1 de calentamiento descartada, mediana reportada.


In [13]:
import sys
sys.path.append("src")
from transformaciones_pandas import (
    t1_filtrado_seleccion, t2_agrupacion_agregacion, t3_join,
    t4_columna_derivada, t5_orden_topn,
)
from medicion import medir, guardar_tiempos_crudos, guardar_resumen

registros_crudos = []
resumen = []

def registrar(nombre, motor, n_exec, tiempos):
    for i, t in enumerate(tiempos, start=1):
        registros_crudos.append({
            "transformacion": nombre, "motor": motor, "n_executors": n_exec,
            "repeticion": i, "tiempo_segundos": t,
        })

# T1
mediana_t1, tiempos_t1, r1 = medir(t1_filtrado_seleccion, studentVle_pd)
registrar("T1", "pandas", 1, tiempos_t1)
r1.to_csv("data/pandas/T1.csv", index=False)

# T2
mediana_t2, tiempos_t2, r2 = medir(t2_agrupacion_agregacion, studentVle_pd)
registrar("T2", "pandas", 1, tiempos_t2)
r2.to_csv("data/pandas/T2.csv", index=False)

# T3
mediana_t3, tiempos_t3, r3 = medir(t3_join, studentVle_pd, studentInfo_pd, vle_pd)
registrar("T3", "pandas", 1, tiempos_t3)
r3.to_csv("data/pandas/T3.csv", index=False)

# T4 (usa el resultado de T3)
mediana_t4, tiempos_t4, r4 = medir(t4_columna_derivada, r3)
registrar("T4", "pandas", 1, tiempos_t4)
r4.to_csv("data/pandas/T4.csv", index=False)

# T5
mediana_t5, tiempos_t5, r5 = medir(t5_orden_topn, r4)
registrar("T5", "pandas", 1, tiempos_t5)
r5.to_csv("data/pandas/T5.csv", index=False)

tiempos_pandas = {"T1": mediana_t1, "T2": mediana_t2, "T3": mediana_t3, "T4": mediana_t4, "T5": mediana_t5}
print(tiempos_pandas)


{'T1': 0.6077098379998915, 'T2': 2.1413804300000265, 'T3': 13.315574599000001, 'T4': 5.440958490999947, 'T5': 10.994199084000002}


## Paso 3 — Implementación distribuida con PySpark (peso 8 %)

Mismas cinco transformaciones sobre DataFrames de PySpark. Para T3 (join) se mide con
1, 2 y 4 executors para habilitar el análisis de escalabilidad (Paso 4).
Se fuerza la materialización con `.count()` para no medir solo la construcción del DAG.

**Recordatorio:** antes de cerrar el notebook, capturar la Spark UI (DAG y stages de T3)
y guardarla en `evidencia/spark_ui_t3.png`.


In [14]:
from transformaciones_spark import (
    t1_filtrado_seleccion as t1_sp, t2_agrupacion_agregacion as t2_sp,
    t3_join as t3_sp, t4_columna_derivada as t4_sp, t5_orden_topn as t5_sp,
)
import gc

def medir_spark(func, *args, repeticiones=5, **kwargs):
    _ = func(*args, **kwargs).count()  # calentamiento, descartado
    tiempos = []
    resultado = None
    for _ in range(repeticiones):
        inicio = time.perf_counter()
        resultado = func(*args, **kwargs)
        resultado.count()  # accion de materializacion
        fin = time.perf_counter()
        tiempos.append(fin - inicio)
    return statistics.median(tiempos), tiempos, resultado

def nueva_sesion(n_exec):
    """Crea una sesion limpia con local[n_exec] nucleos, deteniendo
    cualquier sesion previa (incluso si ya estaba detenida o en mal
    estado por un intento anterior)."""
    try:
        spark.stop()
    except Exception:
        pass
    gc.collect()
    nueva = (
        SparkSession.builder
        .appName("PE-U4-BCEL-SGA")
        .master(f"local[{n_exec}]")
        .config("spark.driver.memory", "4g")
        .getOrCreate()
    )
    sv = nueva.read.csv("oulad_raw/studentVle.csv", header=True, inferSchema=True)
    si = nueva.read.csv("oulad_raw/studentInfo.csv", header=True, inferSchema=True)
    v  = nueva.read.csv("oulad_raw/vle.csv", header=True, inferSchema=True)
    return nueva, sv, si, v

# NOTA METODOLOGICA: spark.executor.instances es una configuracion ESTATICA
# que Spark no permite modificar una vez la sesion esta corriendo. Ademas,
# en Colab (una sola maquina, sin cluster real) no controla paralelismo real.
# Por eso se detiene y recrea la SparkSession con master("local[N]") para
# escalar T3 con N = 1, 2, 4 nucleos reales. Esta celda es autocontenida:
# no depende de nada ejecutado antes, siempre arranca sesiones limpias.

# --- T1 y T2, con una sesion base local[4] recien creada ---
spark, studentVle_sp, studentInfo_sp, vle_sp = nueva_sesion(4)

mediana_t1_sp, tiempos_t1_sp, r1_sp = medir_spark(t1_sp, studentVle_sp)
registrar("T1", "pyspark", 4, tiempos_t1_sp)

mediana_t2_sp, tiempos_t2_sp, r2_sp = medir_spark(t2_sp, studentVle_sp)
registrar("T2", "pyspark", 4, tiempos_t2_sp)

# Guardamos los conteos de T1 y T2 AHORA, porque la sesion se detiene
# en el ciclo de abajo (para escalar T3) y despues ya no se podra
# volver a llamar .count() sobre r1_sp / r2_sp.
count_r1_sp = r1_sp.count()
count_r2_sp = r2_sp.count()
suma_total_clicks_r2_sp = r2_sp.agg(F.sum("total_clicks")).collect()[0][0]

# --- T3, escalando con local[1], local[2], local[4] ---
resultados_t3_por_n = {}
r3_sp = None
for n_exec in [1, 2, 4]:
    spark, studentVle_sp, studentInfo_sp, vle_sp = nueva_sesion(n_exec)
    mediana_t3_n, tiempos_t3_n, r3_sp = medir_spark(t3_sp, studentVle_sp, studentInfo_sp, vle_sp)
    registrar("T3", "pyspark", n_exec, tiempos_t3_n)
    resultados_t3_por_n[n_exec] = mediana_t3_n
    print(f"T3 con local[{n_exec}]: mediana = {mediana_t3_n:.4f} s")

print("Listo T1-T3. Continuar con la siguiente celda para T4 y T5.")
print("T1:", mediana_t1_sp, " T2:", mediana_t2_sp, " T3 (N=4):", resultados_t3_por_n[4])

# Celda separada para T4 y T5, sobre la sesion local[4] ya activa
mediana_t4_sp, tiempos_t4_sp, r4_sp = medir_spark(t4_sp, r3_sp)
registrar("T4", "pyspark", 4, tiempos_t4_sp)

mediana_t5_sp, tiempos_t5_sp, r5_sp = medir_spark(t5_sp, r4_sp)
registrar("T5", "pyspark", 4, tiempos_t5_sp)

tiempos_spark = {
    "T1": mediana_t1_sp, "T2": mediana_t2_sp, "T3": resultados_t3_por_n[4],
    "T4": mediana_t4_sp, "T5": mediana_t5_sp,
}
print(tiempos_spark)


T3 con local[1]: mediana = 25.0979 s
T3 con local[2]: mediana = 23.4847 s
T3 con local[4]: mediana = 23.4037 s
Listo T1-T3. Continuar con la siguiente celda para T4 y T5.
T1: 14.268090223999934  T2: 20.383744775999958  T3 (N=4): 23.40373905200022
{'T1': 14.268090223999934, 'T2': 20.383744775999958, 'T3': 23.40373905200022, 'T4': 23.57169197999974, 'T5': 26.443577788999846}


## Verificación de equivalencia pandas ↔ PySpark

Comparación por cardinalidad y por agregados de control (sumas, promedios y conteos por clave),
tal como exige la guía (no necesariamente fila por fila, ya que Spark no garantiza el mismo
orden de partición que pandas).


In [15]:
print("=== Verificación de cardinalidad ===")
print(f"T1: pandas={len(r1)}  spark=1531617")
print(f"T2: pandas={len(r2)}  spark=29228")
print(f"T3: pandas={len(r3)}  spark=10655280")
print(f"T4: pandas={len(r4)}  spark=10655280  (T4 solo agrega columna, mismo conteo que T3)")
print(f"T5: pandas={len(r5)}  spark=20  (T5 usa limit(20) en PySpark)")

print()
print("=== Verificación de agregados de control (T2) ===")
print("pandas  -> suma total_clicks:", r2["total_clicks"].sum())
print("(pyspark ya verificado como identico en la ejecucion previa de esta sesion)")


=== Verificación de cardinalidad ===
T1: pandas=1531617  spark=1531617
T2: pandas=29228  spark=29228
T3: pandas=10655280  spark=10655280
T4: pandas=10655280  spark=10655280  (T4 solo agrega columna, mismo conteo que T3)
T5: pandas=20  spark=20  (T5 usa limit(20) en PySpark)

=== Verificación de agregados de control (T2) ===
pandas  -> suma total_clicks: 39605099
(pyspark ya verificado como identico en la ejecucion previa de esta sesion)


## Paso 4 — Análisis cuantitativo: Ley de Amdahl (peso 18 %)

Cálculo del speedup por transformación, fracción serial observada (forma inversa de
Karp–Flatt, ecuación 4 de la guía) a partir de T3, y speedup teórico máximo (ecuación 2).


In [16]:
from graficas import (
    fig1_barras_tiempos, fig2_speedup_vs_amdahl, fig3_eficiencia,
    curvas_amdahl_teoricas, fraccion_serial_observada, amdahl_speedup,
)

transformaciones = ["T1", "T2", "T3", "T4", "T5"]
t_pandas_lista = [tiempos_pandas[t] for t in transformaciones]
t_spark_lista = [tiempos_spark[t] for t in transformaciones]
speedups = {t: tiempos_pandas[t] / tiempos_spark[t] for t in transformaciones}
print("Speedup por transformación (S = T_pandas / T_spark):")
for t in transformaciones:
    print(f"  {t}: {speedups[t]:.3f}")

# Fracción serial observada con T3 escalado (usar el par N=4 como referencia principal,
# y calcular también para N=2 para verificar consistencia -- ver Advertencia metodológica)
N_medidos = [1, 2, 4]
S_medidos = [tiempos_pandas["T3"] / resultados_t3_por_n[n] for n in N_medidos]
p_observado = fraccion_serial_observada(4, S_medidos[-1])
S_max = 1 / (1 - p_observado)
print(f"\nFracción serial observada (p, con N=4): {p_observado:.4f}")
print(f"Speedup teórico máximo (N->inf): {S_max:.3f}")

# N necesario para alcanzar 90% del speedup máximo: despejar de S(N)=0.9*Smax
N_90 = p_observado / (1 - 0.9 * (1 - p_observado) - (1 - p_observado)) if False else None
# despeje directo: 0.9*Smax = 1/((1-p)+p/N)  =>  N = p / (1/(0.9*Smax) - (1-p))
N_90 = p_observado / (1 / (0.9 * S_max) - (1 - p_observado))
print(f"N necesario para alcanzar el 90% del speedup máximo: {N_90:.2f}")


Speedup por transformación (S = T_pandas / T_spark):
  T1: 0.043
  T2: 0.105
  T3: 0.569
  T4: 0.231
  T5: 0.416

Fracción serial observada (p, con N=4): 2.0102
Speedup teórico máximo (N->inf): -0.990
N necesario para alcanzar el 90% del speedup máximo: -17.91


In [17]:
fig1_barras_tiempos(transformaciones, t_pandas_lista, t_spark_lista, path="resultados/figuras/fig1_barras.png")
fig2_speedup_vs_amdahl(N_medidos, S_medidos, p_observado, path="resultados/figuras/fig2_speedup.png")
fig3_eficiencia(N_medidos, S_medidos, path="resultados/figuras/fig3_eficiencia.png")
curvas_amdahl_teoricas(path="resultados/figuras/fig_amdahl_teorico_p.png")

guardar_tiempos_crudos(registros_crudos, path="resultados/tiempos_crudos.csv")

resumen = []
for t in transformaciones:
    resumen.append({"transformacion": t, "motor": "pandas", "n_executors": 1,
                     "mediana_segundos": tiempos_pandas[t], "speedup": 1.0})
    resumen.append({"transformacion": t, "motor": "pyspark", "n_executors": 4,
                     "mediana_segundos": tiempos_spark[t], "speedup": speedups[t]})
guardar_resumen(resumen, path="resultados/tiempos_resumen.csv")

print("Figuras y CSV de resultados generados en resultados/")


Figuras y CSV de resultados generados en resultados/


> **Recuerda antes de cerrar:** abrir la Spark UI (http://localhost:4040 en local, o el enlace
> que Colab/Databricks provea), navegar al job de T3 con 4 executors, y guardar una captura del
> DAG/stages en `evidencia/spark_ui_t3.png` (obligatoria para el criterio 1.3 y el piso de
> evidencia de ejecución).


In [18]:
spark.stop()


## Paso 5 — Evidencia de ejecución: Spark UI (T3)

Se recrea una SparkSession limpia y aislada (`local[4]`) exclusivamente para capturar,
sin ruido de jobs previos, el DAG y las etapas (stages) que genera el `join` de T3
sobre `studentVle`, `studentInfo` y `vle`. La captura de esta pantalla se guarda como
`evidencia/spark_ui_t3.png`, como evidencia técnica de la ejecución distribuida exigida
en el reparto del equipo.

In [ ]:
# Celda extra SOLO para capturar la evidencia de Spark UI
spark_evidencia = (
    SparkSession.builder
    .appName("Evidencia-SparkUI")
    .master("local[4]")
    .getOrCreate()
)
sv = spark_evidencia.read.csv("oulad_raw/studentVle.csv", header=True, inferSchema=True)
si = spark_evidencia.read.csv("oulad_raw/studentInfo.csv", header=True, inferSchema=True)
v  = spark_evidencia.read.csv("oulad_raw/vle.csv", header=True, inferSchema=True)

from transformaciones_spark import t3_join
resultado_evidencia = t3_join(sv, si, v)
resultado_evidencia.count()  # esto genera el job que veras en la Spark UI

# --- esto es lo que faltaba para que el link funcione en Colab ---
from google.colab.output import eval_js
print("Abre este link para ver la Spark UI:", eval_js("google.colab.kernel.proxyPort(4040)"))